# Support Vector Machines

In this exercise you will train Support Vector Machine classifiers on biomedical datasets and study two important model-selection questions:

* How should we select the regularization parameter C?
* How should we select the kernel?

## 1. Imports and Dataset

Run the following cell to import libraries and dataset. The dataset includes 30 features, related to the characteristics of a tumor tissue, and the corresponding classification of the tumor as benign or malignant:
* X = dataframe of input features
* y = class (0 = benign; 1 = tumor)
* features = the list of feature names
* class_names = the name of the two classes

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

features = X.columns
class_names = ['benign', 'tumor']
print('Feature names:', features)
print('Number of samples:', X.shape[0])
print('Number of tumor samples:', np.sum(y == 1))

Feature names: Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension'],
      dtype='object')
Number of samples: 569
Number of tumor samples: 357


* Split into training and testing set

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (398, 30)
Test set: (171, 30)


## 2. SVM with linear kernel

* Define a Pipeline that includes the StandardScaler and an SVC model with liner kernel (why is is important to scale the data first ?)
* Perform grid search of the regularization parameter C over the values np.logspace(-4, 4, 9) using 5-fold cross validation. Use balanced_accuracy as scoring metric.
* Report the optimal value of the C parameter and the score of the best performing model
* Report accuracy, balanced accuracy, recall, and precision in the testing set

In [3]:
linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="linear")),
])
param_grid = {"svm__C": np.logspace(-4, 4, 9)}
grid_search = GridSearchCV(
    estimator=linear_svm,
    param_grid=param_grid,
    cv=5,
    scoring="balanced_accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
print("Best C:", grid_search.best_params_["svm__C"])
print("Mean CV balanced accuracy:", grid_search.best_score_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Best C: 0.01
Mean CV balanced accuracy: 0.9729885057471265
Accuracy: 0.9590643274853801
Balanced accuracy: 0.9484521028037383
Recall: 0.9906542056074766
Precision: 0.9464285714285714


## 3. SVM with polynomial kernel

* Repeat the previous point using a polynomial kernel. Test polynomial kernels of degrees 2 or 3.

In [4]:
poly_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="poly")),
])
param_grid = {"svm__C": np.logspace(-4, 4, 9), "svm__degree":[2, 3]}
grid_search = GridSearchCV(
    estimator=poly_svm,
    param_grid=param_grid,
    cv=5,
    scoring="balanced_accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
print("Best C:", grid_search.best_params_["svm__C"])
print("Best polynomial degree:", grid_search.best_params_["svm__degree"])
print("Mean CV balanced accuracy:", grid_search.best_score_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Best C: 1000.0
Best polynomial degree: 3
Mean CV balanced accuracy: 0.9543218390804598
Accuracy: 0.9707602339181286
Balanced accuracy: 0.970356308411215
Recall: 0.9719626168224299
Precision: 0.9811320754716981


## 4. SVM with polynomial kernel

* Repeat the previous point using a gaussian kernel (rbf). Test the following values for the parameter gamma:  np.logspace(-4, 1, 6).

In [5]:
rbf_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf")),
])
param_grid = {"svm__C": np.logspace(-4, 4, 9), "svm__gamma":np.logspace(-4, 1, 6)}
grid_search = GridSearchCV(
    estimator=rbf_svm,
    param_grid=param_grid,
    cv=5,
    scoring="balanced_accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
print("Best C:", grid_search.best_params_["svm__C"])
print("Best gamma:", grid_search.best_params_["svm__gamma"])
print("Mean CV balanced accuracy:", grid_search.best_score_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Best C: 10.0
Best gamma: 0.001
Mean CV balanced accuracy: 0.9729885057471265
Accuracy: 0.9707602339181286
Balanced accuracy: 0.9640771028037383
Recall: 0.9906542056074766
Precision: 0.9636363636363636


* Combine the 3 points above into a unique grid search that automatically select the best kernel and corresponding parameters

In [6]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC()),
])
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": np.logspace(-4, 4, 9),
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": np.logspace(-4, 4, 9),
        "svm__gamma": np.logspace(-4, 1, 6),
    },
    {
        "svm__kernel": ["poly"],
        "svm__C": np.logspace(-4, 4, 9),
        "svm__degree": [2, 3],
        "svm__gamma": ["scale"],
    },
]
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
print("Best parameters:")
print(grid_search.best_params_)
print("Mean CV balanced accuracy:", grid_search.best_score_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Best parameters:
{'svm__C': 0.01, 'svm__kernel': 'linear'}
Mean CV balanced accuracy: 0.9729885057471265
Accuracy: 0.9590643274853801
Balanced accuracy: 0.9484521028037383
Recall: 0.9906542056074766
Precision: 0.9464285714285714


## 5. SVM for a more complicated classification problem

The EEG Eye State dataset is a biomedical signal classification dataset derived from electroencephalography (EEG) recordings. The input features corresponds to 14 EEG signales, the output is a binary classification into open/closed eyes.

* Split data into training and test. Use 50% of data for the training (I suggest this splitting in order to reduce the number of samples in the training set, and consequenly the grid search operation. In case grid search is still too slow, decrease the number of samples in the training set).
* Compare the performances using a linear kernel and and rbf kernel. Use the following parameters:
  * C = \[0.1, 1, 10\]
  * gamma = \[0.01, 0.1\]

In [7]:
from sklearn.datasets import fetch_openml

dataset = fetch_openml(
    name="eeg-eye-state",
    version=1,
    as_frame=True
)
X = dataset.data
y = dataset.target
y = y.astype(int)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (14980, 14)
Shape of y: (14980,)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.50,
    stratify=y,
    random_state=42,
)

In [9]:
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC()),
])
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": [0.1, 1, 10],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": [0.1, 1, 10],
        "svm__gamma": [0.01, 0.1],
    },
]
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
print("Best parameters:")
print(grid_search.best_params_)
print("Mean CV balanced accuracy:", grid_search.best_score_)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
#print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))

Training set: (7490, 14)
Test set: (7490, 14)
Best parameters:
{'svm__C': 10, 'svm__gamma': 0.1, 'svm__kernel': 'rbf'}
Mean CV balanced accuracy: 0.9303316991633629
Accuracy: 0.9448598130841122
Balanced accuracy: 0.9434299696149953
Precision: 0.943436754176611
